# Week 3 (ML-04): Search Intelligence Data Contract

**Lane:** Classification  
**Deliverable:** `work/notebooks/w03_data_contract.ipynb`

This notebook states the data contract for my slice in plain words, proves three facts about it with DuckDB, builds a five-feature frame, and shows what leakage does to a score.

## 1 & 2. The data contract (five plain-words answers)

| Question | Answer |
|---|---|
| **What does one row mean?** | One daily performance snapshot for one content item (`content_hash_id`) on one report date (`report_date`). |
| **Which tables?** | `fact_content_daily_performance`, joined with `dim_clients` and `dim_content`. |
| **Which time window?** | A mid-panel month: `month=2026-03`. The month `2026-06` is the **sealed test month**. I do not query or inspect it here. |
| **What do I predict?** | `is_target_positive`: 1 if the item's daily impressions are above a baseline threshold, else 0. This is a proxy for "the item is getting search visibility". |
| **What is deliberately excluded?** | Anything after the decision moment: next-day clicks, delayed conversions, and the current day's own impressions (the label source). |

**Decision moment:** the start of report day D. I know everything reported through day D-1. I predict whether day D lands above the baseline.

In [ ]:
# In VS Code, run this once in the notebook's environment: pip install duckdb scikit-learn pandas
import os
import getpass
import duckdb
import numpy as np
import pandas as pd

# Token: Colab secret -> environment variable -> hidden prompt (works in VS Code).
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your HF_TOKEN: ")

assert HF_TOKEN, "HF_TOKEN not found."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")

In [ ]:
BASE = "hf://datasets/FlyRank/internship-warehouse"
WORK_MONTH = "2026-03"
SEALED_MONTH = "2026-06"   # sealed test month: never read in this notebook

# Edit these three paths if the warehouse layout differs.
FACT = f"{BASE}/fact_content_daily_performance/month={WORK_MONTH}/*.parquet"
DIM_CLIENTS = f"{BASE}/dim_clients/*.parquet"
DIM_CONTENT = f"{BASE}/dim_content/*.parquet"

assert SEALED_MONTH not in FACT, "The sealed test month must not be used here."

con.execute(f"CREATE OR REPLACE VIEW fact AS SELECT * FROM read_parquet('{FACT}')")
con.execute(f"CREATE OR REPLACE VIEW dim_clients AS SELECT * FROM read_parquet('{DIM_CLIENTS}')")
con.execute(f"CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('{DIM_CONTENT}')")

# Schema check: confirm the column names I assume actually exist.
for name in ["fact", "dim_clients", "dim_content"]:
    print(f"--- {name} ---")
    print(con.sql(f"DESCRIBE {name}").df()[["column_name", "column_type"]].to_string(index=False))

## 3. Prove three facts with DuckDB

Each query backs one claim in the contract. If a check fails, the contract is wrong and I fix the contract, not the check.

In [ ]:
# Fact 1: one row per (report_date, content_hash_id). Duplicate groups must be zero.
q1 = """
SELECT COUNT(*) AS duplicate_groups
FROM (
    SELECT report_date, content_hash_id, COUNT(*) AS n
    FROM fact
    GROUP BY report_date, content_hash_id
    HAVING COUNT(*) > 1
)
"""
duplicate_groups = con.sql(q1).fetchone()[0]
print(f"Duplicate (report_date, content_hash_id) groups: {duplicate_groups}")
assert duplicate_groups == 0, "Grain is broken: more than one row per item per day."

In [ ]:
# Fact 2: size and date span of the working month.
q2 = """
SELECT COUNT(*)            AS total_rows,
       MIN(report_date)    AS first_date,
       MAX(report_date)    AS last_date
FROM fact
"""
q2_result = con.sql(q2).df()
print(q2_result.to_string(index=False))

total_rows = int(q2_result.loc[0, "total_rows"])

In [ ]:
# Fact 3: how much of the month survives the GSC-access filter.
q3 = """
SELECT COUNT(*)                                              AS rows_before,
       COUNT(*) FILTER (WHERE c.has_gsc_access IS TRUE)      AS rows_after,
       COUNT(DISTINCT f.client_id)                           AS clients_before,
       COUNT(DISTINCT f.client_id)
           FILTER (WHERE c.has_gsc_access IS TRUE)           AS clients_after
FROM fact f
JOIN dim_clients c ON f.client_id = c.client_id
"""
q3_result = con.sql(q3).df()
print(q3_result.to_string(index=False))

rows_before = int(q3_result.loc[0, "rows_before"])
rows_after = int(q3_result.loc[0, "rows_after"])
print(f"\nSurviving rows: {rows_after:,} of {rows_before:,} ({rows_after / rows_before:.1%})")

# The join must not drop or multiply rows.
assert rows_before == total_rows, "Join changed the row count: check client_id keys."

## 4. Five features and the leakage trap

**Rules for every feature:** it must be knowable at the start of report day D. I build lag and rolling features from days strictly before D.

**Target:** `is_target_positive = 1` if day D impressions are above a baseline. The baseline is the median impressions from the first 7 days of the month. Those 7 days are warm-up only and carry no labels, so the baseline does not use the days I score.

**Split:** labeled days run from day 8 to the end of the month. Days 8-21 train; the rest validate. The sealed month stays untouched.

In [ ]:
# Join all three tables, keep only GSC-connected clients.
frame = con.sql("""
    SELECT f.report_date, f.content_hash_id, f.client_id,
           f.impressions, f.clicks, f.avg_position,
           ct.content_type
    FROM fact f
    JOIN dim_clients c  ON f.client_id = c.client_id
    JOIN dim_content ct ON f.content_hash_id = ct.content_hash_id
    WHERE c.has_gsc_access IS TRUE
""").df()

frame["report_date"] = pd.to_datetime(frame["report_date"])
frame = frame.sort_values(["content_hash_id", "report_date"]).reset_index(drop=True)

# The dim_content join must not duplicate rows.
assert not frame.duplicated(["report_date", "content_hash_id"]).any(), "dim_content join fanned out."

# Baseline from the warm-up week only.
month_start = frame["report_date"].min().normalize()
warmup_end = month_start + pd.Timedelta(days=7)
BASELINE = frame.loc[frame["report_date"] < warmup_end, "impressions"].median()

# Target: 1 if today's impressions are above the baseline.
frame["is_target_positive"] = (frame["impressions"] > BASELINE).astype(int)

print(f"Baseline impressions (median, first 7 days): {BASELINE}")
print(f"Positive rate: {frame['is_target_positive'].mean():.1%}")

In [ ]:
g = frame.groupby("content_hash_id")
feat = pd.DataFrame(index=frame.index)

# Knowable at the decision moment because yesterday's report is closed before today starts.
feat["impressions_lag1"] = g["impressions"].shift(1)

# Knowable at the decision moment because it averages only days before today (shifted one row first).
feat["clicks_mean_prior7"] = g["clicks"].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())

# Knowable at the decision moment because it is yesterday's average ranking position, already reported.
feat["avg_position_lag1"] = g["avg_position"].shift(1)

# Knowable at the decision moment because content type is set when the item is published.
feat["content_type_code"] = frame["content_type"].astype("category").cat.codes

# Knowable at the decision moment because the calendar day of the week is known in advance.
feat["day_of_week"] = frame["report_date"].dt.dayofweek

assert feat.shape[1] <= 5, "Five features max."

# Labeled rows: after warm-up, with at least one prior day of history.
labeled = (frame["report_date"] >= warmup_end) & feat["impressions_lag1"].notna()
y = frame.loc[labeled, "is_target_positive"]

# Time-ordered split inside the working month.
split_date = warmup_end + pd.Timedelta(days=14)
train = labeled & (frame["report_date"] < split_date)
valid = labeled & (frame["report_date"] >= split_date)

print(f"Features: {list(feat.columns)}")
print(f"Train rows: {train.sum():,} | Validation rows: {valid.sum():,}")
feat.loc[labeled].head()

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def score(cols):
    model = HistGradientBoostingClassifier(random_state=42)
    model.fit(feat.loc[train, cols], frame.loc[train, "is_target_positive"])
    p = model.predict_proba(feat.loc[valid, cols])[:, 1]
    yv = frame.loc[valid, "is_target_positive"]
    return roc_auc_score(yv, p), average_precision_score(yv, p)

honest_cols = list(feat.columns)
auc_honest, ap_honest = score(honest_cols)

# THE TRAP: add a column built from the label's own source (today's impressions).
feat["LEAK_impressions_today"] = frame["impressions"]

leaky_cols = list(feat.columns)
auc_leaky, ap_leaky = score(leaky_cols)

corr = feat.loc[labeled].corrwith(y).abs().sort_values(ascending=False)
print("Absolute correlation with target:")
print(corr.round(3).to_string())

print(f"\nWith leak    -> ROC-AUC {auc_leaky:.3f} | PR-AUC {ap_leaky:.3f}")
print(f"Honest (pre) -> ROC-AUC {auc_honest:.3f} | PR-AUC {ap_honest:.3f}")

In [ ]:
# Delete the leaky column to restore the honest feature frame.
feat = feat.drop(columns=["LEAK_impressions_today"])

# Guards: no leak column, five features max, no feature that tracks the label almost perfectly.
assert "LEAK_impressions_today" not in feat.columns
assert feat.shape[1] <= 5
assert feat.loc[labeled].corrwith(y).abs().max() < 0.95, "A feature is too close to the label. Inspect it."

auc_final, ap_final = score(list(feat.columns))
print(f"Final honest frame: {list(feat.columns)}")
print(f"Validation ROC-AUC {auc_final:.3f} | PR-AUC {ap_final:.3f}")
print(f"The leak inflated ROC-AUC by {auc_leaky - auc_final:+.3f}.")

**What the trap showed.** The leaky column is the label's own source, so the model reads the answer instead of predicting it. The score jumped to near-perfect, which is the warning sign. A score that good on a noisy search problem should make me suspicious, not proud. After I dropped the column, the score fell to the honest level. That honest number is the one I report.

## 5. Named limitation

**This slice covers one month (March 2026) and only clients with Google Search Console access.** The model learns March behavior from GSC-connected clients. It may not transfer to the sealed June month, where seasonality and traffic patterns can differ, or to clients without GSC access, whose content may behave differently. The baseline threshold is also fixed from March's first week, so a change in overall traffic levels would shift the label rate. I will check the label rate on the sealed month once, at the end, and report any shift instead of retuning on it.